In [1]:
import pandas as pd
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.config import MERGED_DATA
def statement_to_dict_df(file, statement_name):
    ticker = Path(file).stem.replace(f"_{statement_name}", "")

    df = pd.read_csv(file)

    rows = []

    # First column contains metric names
    metric_col = df.columns[0]

    # Remaining columns are dates
    for date in df.columns[1:]:
        metrics = (
            df[[metric_col, date]]
            .dropna()
            .set_index(metric_col)[date]
            .to_dict()
        )

        rows.append({
            "company": ticker,
            "date": date,
            statement_name: metrics
        })

    return pd.DataFrame(rows)

In [2]:
from pathlib import Path

DATA_DIR = Path("../data")
BALANCE_SHEET_DIR = DATA_DIR / "balance_sheet"
CASH_FLOW_DIR = DATA_DIR / "cashflows"
FINANCIALS_DIR = DATA_DIR / "financials"
DATA_TICKER = DATA_DIR / "processed" / "layoffs_with_tickers.csv"
tickers = set()

for file in BALANCE_SHEET_DIR.glob("*.csv"):
    ticker = file.stem.replace("_balancesheet", "")
    tickers.add(ticker)
 
print(f"Found {len(tickers)} companies")

Found 7665 companies


In [3]:
master_rows = []

for ticker in tickers:

    dfs = []

    statement_info = [
        ("balancesheet", BALANCE_SHEET_DIR, "_balancesheet.csv"),
        ("cashflow", CASH_FLOW_DIR, "_cashflow.csv"),
        ("financials", FINANCIALS_DIR, "_financials.csv"),
    ]

    for statement_name, directory, suffix in statement_info:

        file = directory / f"{ticker}{suffix}"

        if not file.exists():
            continue

        try:
            statement_df = statement_to_dict_df(
                file,
                statement_name
            )

            if not statement_df.empty:
                dfs.append(statement_df)

        except Exception as e:
            print(
                f"Failed {statement_name} "
                f"for {ticker}: {e}"
            )

    # Skip companies with no statements
    if len(dfs) == 0:
        continue

    # Merge all available statements
    company_df = dfs[0]

    for df in dfs[1:]:
        company_df = company_df.merge(
            df,
            on=["company", "date"],
            how="outer"
        )

    master_rows.append(company_df)

    print(f"Processed {ticker}")

Processed CBIO
Processed 9563.T
Processed CAPS
Processed PAA
Processed ESCA
Processed BAYAU
Processed RKDA
Processed NNDM
Processed GCMG
Processed IBIO
Processed IRIX
Processed COHR
Processed 002222.SZ
Processed MCTA
Processed CASH
Processed DNTH
Processed ELDN
Processed QUIK
Processed FROG
Processed IPSC
Processed FTRK
Processed MQ
Processed BYFC
Processed NAMSW
Processed HCTI
Processed SHFS
Processed AMCR
Processed MNSBP
Processed 301231.SZ
Processed LICN
Processed DYORW
Processed KFFB
Processed COCP
Processed ILKAY
Processed WFRD
Processed NAGE
Processed SINT
Processed SBR
Processed MOB
Processed GNTA
Processed AI
Processed DUK
Processed LIXT
Processed PNRG
Processed LASE
Processed Z88.F
Processed L360.F
Processed SLNH
Processed QUAD
Processed MANH
Processed ORIQ
Processed OSPN
Processed PFAI
Processed FNWD
Processed BI1.DU
Processed CINF
Processed NRIS
Processed GGEO.ST
Processed EDUC
Processed CMND
Processed 5388.TW
Processed LSTA
Processed GCTK
Processed CLNN
Processed MACIU
Proc

In [4]:
import pandas as pd

dataset = pd.concat(master_rows, ignore_index=True)
dataset["date"] = pd.to_datetime(dataset["date"])

dataset["quarter"] = dataset["date"].dt.to_period("Q").astype(str)

bs_features = pd.json_normalize(
    dataset["balancesheet"]
).add_prefix("bs_")


cf_features = pd.json_normalize(
    dataset["cashflow"]
).add_prefix("cf_")

fin_features = pd.json_normalize(
    dataset["financials"]
).add_prefix("fin_")

features_df = pd.concat(
    [
        dataset[["company", "date", "quarter"]],
        bs_features,
        cf_features,
        fin_features,
    ],
    axis=1,
)
# features_df.fillna("NaN")
# print(features_df.fillna("NaN").head())

features_df.to_csv(
    MERGED_DATA["MERGED_OUTPUT_CSV_PATH"],
    index=False
)

# Verify
print(features_df.shape)
print(features_df.head())

(30015, 350)
  company       date quarter  bs_Ordinary Shares Number  bs_Share Issued  \
0    CBIO 2024-12-31  2024Q4                        NaN              NaN   
1    CBIO 2025-03-31  2025Q1                        NaN              NaN   
2    CBIO 2025-06-30  2025Q2                 16782562.0       16782562.0   
3    CBIO 2025-09-30  2025Q3                 16782516.0       16782516.0   
4    CBIO 2025-12-31  2025Q4                 30446767.0       30446767.0   

   bs_Total Debt  bs_Tangible Book Value  bs_Invested Capital  \
0            NaN                     NaN                  NaN   
1            NaN                     NaN                  NaN   
2      1644000.0             135254000.0          135254000.0   
3      1741000.0             112641000.0          112641000.0   
4      1644000.0             199012000.0          199012000.0   

   bs_Working Capital  bs_Net Tangible Assets  ...  fin_Other Taxes  \
0                 NaN                     NaN  ...              NaN 

## No-layoff companies

Same pipeline as above, but restricted to tickers from `no_layoff_tickers.csv`, cross-checked against the layoff tickers to guarantee no overlap. Output goes to `MERGED_DATA["MERGED_OUTPUT_NO_LAYOFF_CSV_PATH"]`.

In [5]:
NO_LAYOFF_TICKERS_CSV = DATA_DIR / "processed" / "no_layoff_tickers.csv"

no_layoff_df = pd.read_csv(NO_LAYOFF_TICKERS_CSV)
no_layoff_tickers = set(no_layoff_df["Symbol"])

layoff_tickers_df = pd.read_csv(DATA_TICKER)
layoff_tickers = set(layoff_tickers_df["Ticker"])

# Cross-check: drop any ticker that also appears in the layoff dataset
overlap = no_layoff_tickers & layoff_tickers
if overlap:
    print(f"Removing {len(overlap)} tickers that overlap with layoff companies")
    no_layoff_tickers -= overlap

# Keep only tickers that actually have at least one statement file
no_layoff_tickers = {
    ticker for ticker in no_layoff_tickers
    if (BALANCE_SHEET_DIR / f"{ticker}_balancesheet.csv").exists()
    or (CASH_FLOW_DIR / f"{ticker}_cashflow.csv").exists()
    or (FINANCIALS_DIR / f"{ticker}_financials.csv").exists()
}

print(f"Found {len(no_layoff_tickers)} no-layoff companies with statement data")

Removing 2 tickers that overlap with layoff companies
Found 4960 no-layoff companies with statement data


In [6]:
master_rows_no_layoff = []

for ticker in no_layoff_tickers:

    dfs = []

    statement_info = [
        ("balancesheet", BALANCE_SHEET_DIR, "_balancesheet.csv"),
        ("cashflow", CASH_FLOW_DIR, "_cashflow.csv"),
        ("financials", FINANCIALS_DIR, "_financials.csv"),
    ]

    for statement_name, directory, suffix in statement_info:

        file = directory / f"{ticker}{suffix}"

        if not file.exists():
            continue

        try:
            statement_df = statement_to_dict_df(
                file,
                statement_name
            )

            if not statement_df.empty:
                dfs.append(statement_df)

        except Exception as e:
            print(
                f"Failed {statement_name} "
                f"for {ticker}: {e}"
            )

    # Skip companies with no statements
    if len(dfs) == 0:
        continue

    # Merge all available statements
    company_df = dfs[0]

    for df in dfs[1:]:
        company_df = company_df.merge(
            df,
            on=["company", "date"],
            how="outer"
        )

    master_rows_no_layoff.append(company_df)

print(f"Processed {len(master_rows_no_layoff)} no-layoff companies")

Processed 3443 no-layoff companies


In [7]:
dataset_no_layoff = pd.concat(master_rows_no_layoff, ignore_index=True)
dataset_no_layoff["date"] = pd.to_datetime(dataset_no_layoff["date"])

dataset_no_layoff["quarter"] = dataset_no_layoff["date"].dt.to_period("Q").astype(str)

bs_features_nl = pd.json_normalize(
    dataset_no_layoff["balancesheet"]
).add_prefix("bs_")

cf_features_nl = pd.json_normalize(
    dataset_no_layoff["cashflow"]
).add_prefix("cf_")

fin_features_nl = pd.json_normalize(
    dataset_no_layoff["financials"]
).add_prefix("fin_")

features_df_no_layoff = pd.concat(
    [
        dataset_no_layoff[["company", "date", "quarter"]],
        bs_features_nl,
        cf_features_nl,
        fin_features_nl,
    ],
    axis=1,
)

features_df_no_layoff.to_csv(
    MERGED_DATA["MERGED_OUTPUT_NO_LAYOFF_CSV_PATH"],
    index=False
)

# Verify
print(features_df_no_layoff.shape)
print(features_df_no_layoff.head())

(17983, 349)
  company       date quarter  bs_Ordinary Shares Number  bs_Share Issued  \
0    CBIO 2024-12-31  2024Q4                        NaN              NaN   
1    CBIO 2025-03-31  2025Q1                        NaN              NaN   
2    CBIO 2025-06-30  2025Q2                 16782562.0       16782562.0   
3    CBIO 2025-09-30  2025Q3                 16782516.0       16782516.0   
4    CBIO 2025-12-31  2025Q4                 30446767.0       30446767.0   

   bs_Total Debt  bs_Tangible Book Value  bs_Invested Capital  \
0            NaN                     NaN                  NaN   
1            NaN                     NaN                  NaN   
2      1644000.0             135254000.0          135254000.0   
3      1741000.0             112641000.0          112641000.0   
4      1644000.0             199012000.0          199012000.0   

   bs_Working Capital  bs_Net Tangible Assets  ...  \
0                 NaN                     NaN  ...   
1                 NaN          